In [ ]:
import os 
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader 
from langchain_text_splitters import RecursiveCharacterTextSplitter 
#from langchain.text_splitter import RecursiveCharacterTextSplitter 
from pathlib import Path

In [13]:
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader


def process_all_pdfs(pdf_directory):

    all_documents = []

    pdf_dir = Path(pdf_directory)

    pdf_files = list(
        pdf_dir.glob("**/*.pdf")
    )

    print(
        f"Found {len(pdf_files)} PDFs"
    )

    for pdf_file in pdf_files:

        try:

            loader = PyPDFLoader(
                str(pdf_file)
            )

            documents = loader.load()

            total_pages = len(documents)

            for page_num, doc in enumerate(documents):

                doc.metadata.update({

                    "source_file":
                    pdf_file.name,

                    "full_path":
                    str(pdf_file),

                    "page":
                    page_num + 1,

                    "total_pages":
                    total_pages,

                    "file_type":
                    "pdf"

                })

            all_documents.extend(
                documents
            )

            print(
                f"✓ {pdf_file.name} "
                f"({total_pages} pages)"
            )

        except Exception as e:

            print(
                f"✗ {pdf_file.name}"
            )

            print(e)

    print(
        f"\nLoaded "
        f"{len(all_documents)} pages"
    )

    return all_documents
all_pdf_documents = process_all_pdfs("../data/raw")



Found 5 PDFs
✓ Attention Is All You Nee.pdf (15 pages)
✓ Lost in the Middle How Language Models Use Long Contexts.pdf (18 pages)
✓ REACT SYNERGIZING REASONING AND ACTING IN.pdf (33 pages)
✓ Retrieval-Augmented Generation for.pdf (19 pages)
✓ Toolformer Language Models Can Teach Themselves to Use Tool.pdf (17 pages)

Loaded 102 pages


In [14]:
print(all_pdf_documents)

[Document(metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2024-04-10T21:11:43+00:00', 'author': '', 'keywords': '', 'moddate': '2024-04-10T21:11:43+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': '..\\data\\raw\\Attention Is All You Nee.pdf', 'total_pages': 15, 'page': 1, 'page_label': '1', 'source_file': 'Attention Is All You Nee.pdf', 'full_path': '..\\data\\raw\\Attention Is All You Nee.pdf', 'file_type': 'pdf'}, page_content='Provided proper attribution is provided, Google hereby grants permission to\nreproduce the tables and figures in this paper solely for use in journalistic or\nscholarly works.\nAttention Is All You Need\nAshish Vaswani∗\nGoogle Brain\navaswani@google.com\nNoam Shazeer∗\nGoogle Brain\nnoam@google.com\nNiki Parmar∗\nGoogle Research\nnikip@google.com\nJakob Uszkoreit∗\nGoogle Research\nusz@googl

In [25]:
"""
Improved PDF cleaning utilities for RAG pipelines.

Key fixes vs. the naive version:
1. Order of operations matters: strip headers/footers and de-hyphenate
   BEFORE collapsing whitespace, otherwise line-anchored regex (^...$)
   becomes useless once newlines are gone.
2. De-hyphenates words broken across line wraps ("atten-\ntion" -> "attention").
3. Normalizes unicode (ligatures like ﬁ/ﬂ, smart quotes, non-breaking spaces).
4. Strips repeated headers/footers (running titles, arXiv IDs, conference
   footers) by detecting lines that repeat across many pages.
5. Removes reference-citation noise like "[ 35, 2, 5]" -> "[35, 2, 5]"
   (optional - keep if you want citations searchable, strip if not).
6. Filters out boilerplate-only chunks (e.g. a page that's just a figure
   caption with no real content) so you don't pollute your vector store.
7. Works generically - not hardcoded to this one paper.
"""

import re
import unicodedata
from collections import Counter
from pathlib import Path

from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter


# ---------- 1. Unicode / ligature normalization ----------

LIGATURE_MAP = {
    "\ufb00": "ff", "\ufb01": "fi", "\ufb02": "fl",
    "\ufb03": "ffi", "\ufb04": "ffl",
    "\u2019": "'", "\u2018": "'",
    "\u201c": '"', "\u201d": '"',
    "\u2013": "-", "\u2014": "-",
    "\u00a0": " ",  # non-breaking space
}


def normalize_unicode(text: str) -> str:
    text = unicodedata.normalize("NFKC", text)
    for bad, good in LIGATURE_MAP.items():
        text = text.replace(bad, good)
    return text


# ---------- 2. De-hyphenate words broken across line wraps ----------

def dehyphenate(text: str) -> str:
    # "exam-\nple" or "exam- \nple" -> "example"
    text = re.sub(r"(\w)-\s*\n\s*(\w)", r"\1\2", text)
    return text


# ---------- 3. Detect & strip repeated headers/footers ----------

def find_repeated_lines(pages_text: list[str], min_repeat: int = 3) -> set[str]:
    """Lines that appear near-identically on many pages are almost
    certainly running headers/footers, not content."""
    line_counts = Counter()
    for text in pages_text:
        # only look at first/last 2 lines of each page - headers/footers live there
        lines = [l.strip() for l in text.split("\n") if l.strip()]
        candidates = lines[:2] + lines[-2:]
        for line in candidates:
            # normalize page numbers out before counting
            norm = re.sub(r"\d+", "#", line)
            if 0 < len(norm) < 80:
                line_counts[norm] += 1

    return {line for line, count in line_counts.items() if count >= min_repeat}


def strip_known_boilerplate(text: str, repeated_norms: set[str]) -> str:
    lines = text.split("\n")
    keep = []
    for line in lines:
        norm = re.sub(r"\d+", "#", line.strip())
        if norm in repeated_norms:
            continue
        keep.append(line)
    return "\n".join(keep)


# ---------- 4. Standalone page-number / bare-line stripping ----------

def strip_standalone_page_numbers(text: str) -> str:
    # operates line-by-line, BEFORE whitespace collapse
    lines = text.split("\n")
    return "\n".join(l for l in lines if not re.fullmatch(r"\s*\d{1,4}\s*", l))


# ---------- 5. Citation / reference noise cleanup ----------

def fix_citation_spacing(text: str) -> str:
    # "[ 35, 2, 5]" -> "[35, 2, 5]"; "[ 9 ]" -> "[9]"
    text = re.sub(r"\[\s+", "[", text)
    text = re.sub(r"\s+\]", "]", text)
    return text


# ---------- 6. Email / URL stripping ----------

def strip_emails_urls(text: str) -> str:
    text = re.sub(r"\S+@\S+\.\S+", "", text)
    text = re.sub(r"https?://\S+", "", text)
    return text


# ---------- 7. Final whitespace collapse (do this LAST) ----------

def collapse_whitespace(text: str) -> str:
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def strip_footnote_markers(text: str) -> str:
    # author-list markers like ∗ † ‡ §, and stray numeric footnote refs
    # attached directly to words e.g. "Brain4" -> "Brain"
    text = re.sub(r"[∗†‡§¶]", "", text)
    return text


def strip_author_affiliation_block(text: str) -> str:
    """Strip the author-names/affiliations block typically found between
    the title and the Abstract on a paper's first page. Purely structural:
    we drop short, punctuation-less lines immediately preceding 'Abstract'
    (names, affiliations, emails) regardless of institution name, so this
    works for any paper without a hardcoded list of universities/labs."""
    if "Abstract" not in text:
        return text  # only apply on pages containing the abstract heading

    head, _, rest = text.partition("Abstract")
    lines = head.split("\n")

    # Walk backward from just before "Abstract", dropping short lines with
    # no terminal punctuation (the hallmark of a name/affiliation line vs.
    # a real sentence) until we hit prose or run out of lines.
    kept = []
    affiliation_run = True
    for line in reversed(lines):
        stripped = line.strip()
        is_short = 0 < len(stripped) < 60
        no_terminal_punct = stripped and not stripped.endswith((".", ":", ")", "?"))

        if affiliation_run and is_short and no_terminal_punct:
            continue  # drop it - looks like a name/affiliation/email line
        else:
            affiliation_run = False
            kept.append(line)

    kept.reverse()
    new_head = "\n".join(kept)
    return f"{new_head}\nAbstract{rest}"


def trim_metadata(metadata: dict) -> dict:
    """Keep only fields useful for retrieval/citation; drop PDF producer noise."""
    keep_keys = {"source_file", "page", "page_label", "total_pages", "chunk_id", "chunk_index", "section"}
    return {k: v for k, v in metadata.items() if k in keep_keys}


def tag_reference_sections(documents) -> None:
    """Mark pages that fall within a References/Bibliography section so
    these chunks can be filtered out of retrieval later without deleting
    them outright (in case citation lookup is ever useful)."""
    in_references = False
    for doc in documents:
        if re.search(r"^\s*References\b|^\s*Bibliography\b", doc.page_content, re.MULTILINE):
            in_references = True
        doc.metadata["section"] = "references" if in_references else "body"


def dedup_chunks(chunks, similarity_threshold: float = 0.9):
    """Drop chunks that are near-duplicates of a previous chunk
    (common artifact of recursive splitting on short pages)."""
    from difflib import SequenceMatcher

    kept = []
    for chunk in chunks:
        is_dup = False
        for prev in kept[-5:]:  # only compare against recent chunks for speed
            ratio = SequenceMatcher(None, chunk.page_content, prev.page_content).ratio()
            if ratio >= similarity_threshold:
                is_dup = True
                break
        if not is_dup:
            kept.append(chunk)
    print(f"Deduped: {len(kept)}/{len(chunks)} chunks kept")
    return kept


# ---------- Pipeline ----------

def clean_page_text(raw_text: str, repeated_norms: set[str]) -> str:
    text = normalize_unicode(raw_text)
    text = strip_known_boilerplate(text, repeated_norms)
    text = strip_standalone_page_numbers(text)
    text = dehyphenate(text)
    text = strip_emails_urls(text)
    text = strip_footnote_markers(text)
    text = strip_author_affiliation_block(text)
    text = fix_citation_spacing(text)
    text = collapse_whitespace(text)
    return text







def figure_table_dominance_ratio(text: str) -> float:
    """Fraction of the chunk's characters that are inside 'Figure N ...'
    or 'Table N ...' caption-style references, vs. prose discussing them.
    A chunk that's mostly caption fragments has low retrieval value;
    a chunk that mentions a figure in passing within real prose is fine."""
    caption_chars = sum(len(m) for m in re.findall(
        r"(?:Figure|Table|Fig\.)\s*\d+[:.]?\s*[^.]{0,150}\.", text))
    return caption_chars / max(len(text), 1)


FIGURE_TABLE_REF = re.compile(r"\b(Figure|Fig\.|Table)\s+\d+\b")


def is_figure_table_reference_only(text: str, max_ratio: float = 0.4) -> bool:
    """Drop chunks where a large fraction of sentences are just pointers
    to a figure/table ('Figure 2 shows...', 'Table 3 demonstrates...')
    without enough surrounding explanatory content to be useful alone."""
    sentences = re.split(r"(?<=[.!?])\s+", text)
    if not sentences:
        return False
    ref_sentences = sum(1 for s in sentences if FIGURE_TABLE_REF.search(s))
    return (ref_sentences / len(sentences)) > max_ratio and len(sentences) <= 6


def is_low_value_chunk(text: str, min_words: int = 20) -> bool:
    """Filter out chunks that are mostly figure axis labels, table
    fragments, or other near-meaningless content for embeddings."""
    words = text.split()
    if len(words) < min_words:
        return True
    alpha_chars = sum(c.isalpha() for c in text)
    if alpha_chars / max(len(text), 1) < 0.5:
        return True
    if is_garbled_text(text):
        return True
    if is_figure_table_reference_only(text):
        return True
    if figure_table_dominance_ratio(text) > 0.5:
        return True
    return False


def is_garbled_text(text: str, threshold: float = 0.25) -> bool:
    """Detect text from mis-encoded/custom-subset fonts (common in PDFs
    with embedded screenshots or special glyph mappings). Such text is
    mostly alphabetic so the alpha-ratio check above won't catch it -
    instead we check for plausible English word shapes (vowel presence,
    reasonable word length, reasonable consonant runs) since garbled
    output tends to produce consonant clusters and words with no vowels."""
    words = re.findall(r"[A-Za-z]{2,}", text)
    if len(words) < 5:
        return False  # too short to judge reliably, let other filters decide

    bad = 0
    for w in words:
        has_vowel = bool(re.search(r"[aeiouAEIOU]", w))
        long_consonant_run = bool(re.search(r"[^aeiouAEIOU\s]{5,}", w))
        if not has_vowel or len(w) > 20 or long_consonant_run:
            bad += 1

    return (bad / len(words)) > threshold


def load_and_clean_pdf(pdf_path: str) -> list:
    """Load and clean a single PDF. Returns [] on failure instead of raising,
    so one bad file doesn't kill a batch run."""
    try:
        loader = PyMuPDFLoader(pdf_path)
        documents = loader.load()
    except Exception as e:
        print(f"  [SKIP] Failed to load {pdf_path}: {e}")
        return []

    if not documents:
        print(f"  [SKIP] No pages extracted from {pdf_path}")
        return []

    source_name = Path(pdf_path).name
    repeated_norms = find_repeated_lines([d.page_content for d in documents])

    cleaned_docs = []
    for doc in documents:
        cleaned = clean_page_text(doc.page_content, repeated_norms)
        if cleaned and not is_low_value_chunk(cleaned, min_words=10):
            doc.page_content = cleaned
            doc.metadata["source_file"] = source_name
            cleaned_docs.append(doc)

    print(f"  Cleaned {len(cleaned_docs)}/{len(documents)} pages from {source_name}")
    tag_reference_sections(cleaned_docs)
    return cleaned_docs


def process_pdf_folder(folder_path: str, chunk_size: int = 1000, chunk_overlap: int = 150, drop_references: bool = True):
    """Process every PDF in a folder. Returns a single combined, deduped
    list of chunks ready for embedding."""
    folder = Path(folder_path)
    pdf_files = sorted(folder.glob("*.pdf"))

    if not pdf_files:
        print(f"No PDFs found in {folder_path}")
        return []

    print(f"Found {len(pdf_files)} PDF(s) in {folder_path}\n")

    all_docs = []
    for pdf_file in pdf_files:
        print(f"Processing: {pdf_file.name}")
        docs = load_and_clean_pdf(str(pdf_file))
        all_docs.extend(docs)

    print(f"\nTotal cleaned pages across all PDFs: {len(all_docs)}")

    # Chunk everything together, then dedup once globally (catches
    # boilerplate/abstract repeats across multiple papers, not just within one)
    all_chunks = chunk_documents(all_docs, chunk_size=chunk_size, chunk_overlap=chunk_overlap, drop_references=drop_references)

    # assign stable chunk_ids now that we have the final global list
    for i, chunk in enumerate(all_chunks):
        chunk.metadata["chunk_id"] = f"{chunk.metadata.get('source_file', 'unknown')}_{i:05d}"
        chunk.metadata["chunk_index"] = i

    return all_chunks


def chunk_documents(documents, chunk_size: int = 1000, chunk_overlap: int = 150, drop_references: bool = True):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", ". ", " ", ""],
    )
    chunks = splitter.split_documents(documents)

    final_chunks = []
    for chunk in chunks:
        if drop_references and chunk.metadata.get("section") == "references":
            continue
        if not is_low_value_chunk(chunk.page_content, min_words=15):
            chunk.metadata = trim_metadata(chunk.metadata)
            final_chunks.append(chunk)

    final_chunks = dedup_chunks(final_chunks)
    print(f"Chunking: {len(final_chunks)}/{len(chunks)} chunks kept after filtering+dedup")
    return final_chunks


if __name__ == "__main__":
    # Point this at a folder containing one or more PDFs
    folder_path = "G:\\langchain project wit RAG\\data\\raw"
    chunks = process_pdf_folder(folder_path)

    print(f"\nFinal chunk count ready for embedding: {len(chunks)}\n")
    for c in chunks[:3]:
        print(c.page_content[:300])
        print(c.metadata)
        print("=" * 50)

Found 5 PDF(s) in G:\langchain project wit RAG\data\raw

Processing: Attention Is All You Nee.pdf
  Cleaned 15/15 pages from Attention Is All You Nee.pdf
Processing: Lost in the Middle How Language Models Use Long Contexts.pdf
  Cleaned 17/18 pages from Lost in the Middle How Language Models Use Long Contexts.pdf
Processing: REACT SYNERGIZING REASONING AND ACTING IN.pdf
  Cleaned 31/33 pages from REACT SYNERGIZING REASONING AND ACTING IN.pdf
Processing: Retrieval-Augmented Generation for.pdf
  Cleaned 19/19 pages from Retrieval-Augmented Generation for.pdf
Processing: Toolformer Language Models Can Teach Themselves to Use Tool.pdf
  Cleaned 17/17 pages from Toolformer Language Models Can Teach Themselves to Use Tool.pdf

Total cleaned pages across all PDFs: 99
Deduped: 304/304 chunks kept
Chunking: 304/435 chunks kept after filtering+dedup

Final chunk count ready for embedding: 304

Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and fi

In [26]:
#from langchain.text_splitter import RecursiveCharacterTextSplitter
import uuid
import hashlib


def split_documents(
    documents,
    chunk_size=500,
    chunk_overlap=50,
    min_chunk_size=100
):
    """
    Production-grade document splitting for RAG.
    """

    # ==========================
    # Validation
    # ==========================
    if not documents:
        raise ValueError(
            "Documents list is empty."
        )

    if chunk_overlap >= chunk_size:
        raise ValueError(
            "chunk_overlap must be smaller "
            "than chunk_size"
        )

    # ==========================
    # Token-aware splitter
    # ==========================
    try:
        text_splitter = (
            RecursiveCharacterTextSplitter
            .from_tiktoken_encoder(
                chunk_size=chunk_size,
                chunk_overlap=chunk_overlap,
                separators=[
                    "\n# ",
                    "\n## ",
                    "\n### ",
                    "\n#### ",
                    "\n\n",
                    "\n",
                    ". ",
                    "! ",
                    "? ",
                    "; ",
                    ", ",
                    " ",
                    ""
                ]
            )
        )

    except Exception:
        print(
            "Warning: tiktoken not found. "
            "Using character splitter."
        )

        text_splitter = (
            RecursiveCharacterTextSplitter(
                chunk_size=chunk_size,
                chunk_overlap=chunk_overlap
            )
        )

    split_docs = text_splitter.split_documents(
        documents
    )

    clean_chunks = []
    seen_hashes = set()

    # ==========================
    # Cleaning + metadata
    # ==========================
    for idx, doc in enumerate(split_docs):

        content = doc.page_content.strip()

        # Remove extra whitespace
        content = " ".join(content.split())

        # Skip empty/small chunks
        if len(content) < min_chunk_size:
            continue

        # Remove duplicate chunks
        content_hash = hashlib.md5(
            content.encode()
        ).hexdigest()

        if content_hash in seen_hashes:
            continue

        seen_hashes.add(content_hash)

        doc.page_content = content

        # Better metadata
        doc.metadata.update({
            "chunk_id":
                str(uuid.uuid4()),

            "chunk_index":
                idx,

            "char_count":
                len(content),

            "word_count":
                len(content.split()),

            "estimated_tokens":
                len(content) // 4
        })

        clean_chunks.append(doc)

    # ==========================
    # Safe stats
    # ==========================
    if clean_chunks:

        chunk_lengths = [
            len(doc.page_content)
            for doc in clean_chunks
        ]

        print("=" * 60)
        print("CHUNKING SUMMARY")
        print("=" * 60)

        print(
            f"Original docs: "
            f"{len(documents)}"
        )

        print(
            f"Chunks created: "
            f"{len(clean_chunks)}"
        )

        print(
            f"Average chunk size: "
            f"{sum(chunk_lengths)//len(chunk_lengths)}"
        )

        print(
            f"Min chunk size: "
            f"{min(chunk_lengths)}"
        )

        print(
            f"Max chunk size: "
            f"{max(chunk_lengths)}"
        )

        print("=" * 60)

        print("\nExample Chunk:")
        print(
            clean_chunks[0]
            .page_content[:200]
        )

        print("\nMetadata:")
        print(
            clean_chunks[0]
            .metadata
        )

    return clean_chunks

In [ ]:
chunks = split_documents(all_documents)
chunks

In [18]:
clean_pdf_text

NameError: name 'clean_pdf_text' is not defined

# chunking

In [ ]:
from langchain_text_splitters import (
    RecursiveCharacterTextSplitter
)
def split_documents(
    documents,
    chunk_size=500,
    chunk_overlap=50
):

    splitter = (
        RecursiveCharacterTextSplitter
        .from_tiktoken_encoder(

            chunk_size=
            chunk_size,

            chunk_overlap=
            chunk_overlap,

            separators=[
                "\n# ",
                "\n## ",
                "\n### ",
                "\n\n",
                "\n",
                ". ",
                "! ",
                "? ",
                " ",
                ""
            ]
        )
    )

    chunks = splitter.split_documents(
        documents
    )

    print(
        f"Created "
        f"{len(chunks)} chunks"
    )

    return chunks

In [ ]:
print(split_documents)

<function split_documents at 0x0000016FC82DD080>


In [ ]:
import uuid
def enrich_metadata(chunks):

    for idx, chunk in enumerate(chunks):

        chunk.metadata.update({

            "chunk_id":
            str(uuid.uuid4()),

            "chunk_index":
            idx,

            "char_count":
            len(
                chunk.page_content
            ),

            "word_count":
            len(
                chunk.page_content.split()
            ),

            "estimated_tokens":
            len(
                chunk.page_content
            ) // 4

        })

    return chunks

### Duplicate check

In [ ]:
import hashlib
def remove_duplicate_chunks(
    chunks
):

    unique_chunks = []

    seen_hashes = set()

    for chunk in chunks:

        content = (
            chunk.page_content
            .strip()
        )

        chunk_hash = (
            hashlib.md5(
                content.encode()
            ).hexdigest()
        )

        if (
            chunk_hash
            in seen_hashes
        ):
            continue

        seen_hashes.add(
            chunk_hash
        )

        unique_chunks.append(
            chunk
        )

    print(
        f"Removed "
        f"{len(chunks)-len(unique_chunks)} "
        f"duplicates"
    )

    return unique_chunks

In [24]:
chunks = split_documents(all_documents)
chunks

NameError: name 'all_documents' is not defined

In [ ]:
for chunk in chunks[:3]:

    print(
        chunk.page_content[:300]
    )

    print(
        chunk.metadata
    )

    print("=" * 50)

Provided proper attribution is provided, Google hereby grants permission to reproduce the tables and figures in this paper solely for use in journalistic or scholarly works. Attention Is All You Need Ashish Vaswani∗ Google Brain Noam Shazeer∗ Google Brain Niki Parmar∗ Google Research Jakob Uszkoreit
{'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2024-04-10T21:11:43+00:00', 'author': '', 'keywords': '', 'moddate': '2024-04-10T21:11:43+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': '..\\data\\raw\\Attention Is All You Nee.pdf', 'total_pages': 15, 'page': 1, 'page_label': '1', 'source_file': 'Attention Is All You Nee.pdf', 'full_path': '..\\data\\raw\\Attention Is All You Nee.pdf', 'file_type': 'pdf', 'chunk_id': 'e7295508-41ed-47eb-98df-4af0515a41de', 'chunk_index': 0, 'char_count': 2288, 'word_count': 332, 'estimated_tokens':

In [ ]:
for i in range(20):
    print(chunks[i].page_content[:500])

Provided proper attribution is provided, Google hereby grants permission to reproduce the tables and figures in this paper solely for use in journalistic or scholarly works. Attention Is All You Need Ashish Vaswani∗ Google Brain Noam Shazeer∗ Google Brain Niki Parmar∗ Google Research Jakob Uszkoreit∗ Google Research Llion Jones∗ Google Research Aidan N. Gomez∗ † University of Toronto Łukasz Kaiser∗ Google Brain Illia Polosukhin∗ ‡ Abstract The dominant sequence transduction models are based on c
. Niki designed, implemented, tuned and evaluated countless model variants in our original codebase and tensor2tensor. Llion also experimented with novel model variants, was responsible for our initial codebase, and efficient inference and visualizations. Lukasz and Aidan spent countless long days designing various parts of and implementing tensor2tensor, replacing our earlier codebase, greatly improving results and massively accelerating our research. †Work performed while at Google Brain. ‡Wo